<a href="https://colab.research.google.com/github/sana-azizz/CVIP/blob/main/%20notebooks/final%20code/Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
%cd /content/CVIP
!ls

/content/CVIP
 CVIP  ' notebooks'   Notebooks   README.md


In [22]:
!git clone https://github.com/sana-azizz/CVIP.git
%cd CVIP

fatal: destination path 'CVIP' already exists and is not an empty directory.
/content/CVIP/CVIP/CVIP


In [21]:
!ls

 CVIP  ' notebooks'   Notebooks   README.md


#Sana Aziz- U25105011
#0)Dataset
#1)LODO
#2)Specular Highlight Error Analysis
#3)Sessile/flat Polyps Error Analysis

#0.#Dataset Preperation


Check overlap in these 2 kvasir datasets


In [ ]:
#have overlap so  omit kvasir seg to avoid data leakage
!pip install -q imagehash
import imagehash
from PIL import Image
from pathlib import Path

def phash_set(folder):
    return {imagehash.phash(Image.open(p).convert("RGB")) for p in Path(folder).glob("*.*")}

h1 = phash_set("/content/drive/MyDrive/Computer Vision Term Project/use dataset/Kvasir SEG/images")
h2 = phash_set("/content/drive/MyDrive/Computer Vision Term Project/use dataset/hyper kvasir/images")
print(f"HyperKvasir: {len(h1)}, Kvasir-SEG: {len(h2)}, phash overlap: {len(h1 & h2)}")

HyperKvasir: 997, Kvasir-SEG: 997, phash overlap: 957


In [ ]:
!pip install -q ultralytics pyyaml pandas pillow


Correct any formating in
Dataset 1,2,3

In [ ]:
import shutil
from pathlib import Path
from PIL import Image

RAW  = Path("/content/drive/MyDrive/Computer Vision Term Project/use dataset")
DEST = Path("/content/drive/MyDrive/Computer Vision Term Project/polyp_datasets")

# 1 & 2: already YOLO format — no changs
for ds in ["CVC-clinicDB", "CVC-colonDB"]:
    src, dst = RAW / ds, DEST / ds
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    n = len(list((dst/"images").glob("*.*")))
    print(f"{ds}: {n} images copied")

# 3: ETIS — convert .tiff to .jpg
src, dst = RAW / "ETIS-LaribPolypDB", DEST / "ETIS-LaribPolypDB"
(dst/"images").mkdir(parents=True, exist_ok=True)
(dst/"labels").mkdir(parents=True, exist_ok=True)

count = 0
for tif in list(src.glob("images/*.tiff")) + list(src.glob("images/*.tif")):
    img = Image.open(tif).convert("RGB")
    out_img = dst / "images" / (tif.stem + ".jpg")
    img.save(out_img, "JPEG", quality=95)
    lbl = src / "labels" / (tif.stem + ".txt")
    if lbl.exists():
        (dst / "labels" / (tif.stem + ".txt")).write_text(lbl.read_text())
    count += 1
print(f"etis: {count} .tiff to .jpg converted")

CVC-clinicDB: 612 images copied
CVC-colonDB: 379 images copied
etis: 156 .tiff to .jpg converted


Correct any formating in
Dataset 4

In [ ]:
import json
from pathlib import Path
from PIL import Image
import shutil

RAW  = Path("/content/drive/MyDrive/Computer Vision Term Project/use dataset/hyper kvasir")
DEST = Path("/content/drive/MyDrive/Computer Vision Term Project//polyp_datasets/hyperkvasir")
(DEST/"images").mkdir(parents=True, exist_ok=True)
(DEST/"labels").mkdir(parents=True, exist_ok=True)

# Find the JSON (could be in root or labels/)
json_candidates = list(RAW.glob("*.json")) + list(RAW.glob("labels/*.json"))
assert json_candidates, " No JSON found — check path"
json_path = json_candidates[0]
print(f"Using JSON: {json_path}")

with open(json_path) as f:
    data = json.load(f)

img_dir = RAW / "images"
count = 0
missing = 0

for key, entry in data.items():
    # Try to find the image: key, key.jpg, key.png etc.
    img_path = None
    for ext in ["", ".jpg", ".png", ".jpeg", ".JPG"]:
        candidate = img_dir / (key + ext)
        if candidate.exists():
            img_path = candidate
            break
    if img_path is None:
        missing += 1
        continue

    W, H = entry["width"], entry["height"]
    lines = []
    for box in entry["bbox"]:
        xmin, ymin = box["xmin"], box["ymin"]
        xmax, ymax = box["xmax"], box["ymax"]
        xc = ((xmin + xmax) / 2) / W
        yc = ((ymin + ymax) / 2) / H
        w  = (xmax - xmin) / W
        h  = (ymax - ymin) / H
        lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

    shutil.copy2(img_path, DEST/"images"/img_path.name)
    (DEST/"labels"/(img_path.stem + ".txt")).write_text("\n".join(lines))
    count += 1

print(f" hyperkvasir: {count} images converted, {missing} missing")

Using JSON: /content/drive/MyDrive/Computer Vision Term Project/use dataset/hyper kvasir/bounding-boxes.json
 hyperkvasir: 1000 images converted, 0 missing


#1.LODO with YOLO11s Model

In [ ]:
#if rerunning need to clear old lodo results
import shutil, os
shutil.rmtree("/content/drive/MyDrive/Computer Vision Term Project/lodo_results/runs", ignore_errors=True)
csv = "/content/drive/MyDrive/Computer Vision Term Project/lodo_results/lodo_summary.csv"
if os.path.exists(csv): os.remove(csv)
print("Cleared")

Cleared


In [ ]:
import shutil
shutil.rmtree("/content/folds", ignore_errors=True)
print("Cleared /content/folds — will rebuild all 5 on next run")

Cleared /content/folds — will rebuild all 5 on next run


In [ ]:
import shutil, os

# Clear old if reruning
csv = "/content/drive/MyDrive/Computer Vision Term Project/lodo_results/lodo_summary.csv"
if os.path.exists(csv): os.remove(csv)
print("Deleted lodo_summary.csv")


runs_dir = "/content/drive/MyDrive/Computer Vision Term Project/lodo_results/runs"
if os.path.exists(runs_dir):
    shutil.rmtree(runs_dir)
    print("Deleted runs/")

Deleted lodo_summary.csv


In [ ]:
# Cell 0 — verify
import torch
assert torch.cuda.is_available(), " GPU not enabled — see Runtime menu"
print(f" GPU: {torch.cuda.get_device_name(0)}")

# Copy data to local (only once per session)
import shutil
from pathlib import Path
SRC = Path("/content/drive/MyDrive/Computer Vision Term Project/polyp_datasets")
LOCAL = Path("/content/polyp_datasets_local")
if not LOCAL.exists():
    shutil.copytree(SRC, LOCAL)
    print(" Data copied to local")
else:
    print(" Local data already present")

 GPU: Tesla T4
 Data copied to local


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q ultralytics pyyaml pandas

In [ ]:


import os, random, yaml
from pathlib import Path
from ultralytics import YOLO
import pandas as pd

BASE    = Path("/content/polyp_datasets_local")

# BASE    = Path("/content/drive/MyDrive/Computer Vision Term Project/polyp_datasets")
RESULTS = Path("/content/drive/MyDrive/Computer Vision Term Project/lodo_results")
FOLDS   = Path("/content/folds")     # ephemeral — safe to lose on disconnect



DATASETS = [
    "CVC-clinicDB",
    "CVC-colonDB",
    "ETIS-LaribPolypDB",
    "hyperkvasir",

]


# ─────────── TRAINING SETTINGS ───────────
MODEL    = "yolo11s.pt"
EPOCHS   = 60
IMGSZ    = 512
BATCH    = 24 #8
PATIENCE = 15
VAL_FRAC = 0.15
SEED     = 0

RESULTS.mkdir(parents=True, exist_ok=True)
FOLDS.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

# ─────────── VERIFY ───────────
print("\nDataset check:")
for ds in DATASETS:
    n_img = len(list((BASE/ds/"images").glob("*.*")))
    n_lbl = len(list((BASE/ds/"labels").glob("*.txt")))
    ok = "correct" if n_img == n_lbl and n_img > 0 else "wrong MISMATCH"
    print(f"  {ok} {ds:22s} {n_img:5d} images  {n_lbl:5d} labels")
print(f"\nRunning {len(DATASETS)}-fold LODO with {MODEL}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

Dataset check:
  correct CVC-clinicDB             612 images    612 labels
  correct CVC-colonDB              379 images    379 labels
  correct ETIS-LaribPolypDB        156 images    156 labels
  correct hyperkvasir             1000 images   1000 labels

Running 4-fold LODO with yolo11s.pt


In [ ]:
import shutil
if os.path.exists("/content/folds"):
    shutil.rmtree("/content/folds")
    print("Deleted /content/folds — will rebuild for 4-dataset pool")

Deleted /content/folds — will rebuild for 4-dataset pool


In [ ]:
SUMMARY_CSV = RESULTS / "lodo_summary.csv"

def make_val_split(train_sets, fold_dir):
    """Create disjoint train + val with dataset-tagged filenames (no collisions)."""
    val_dir      = fold_dir / "val"
    train_only_d = fold_dir / "train_only"

    #if  Already built → skip
    if (val_dir/"images").exists() and (train_only_d/"images").exists() \
       and any((val_dir/"images").iterdir()) and any((train_only_d/"images").iterdir()):
        return

    for d in [val_dir, train_only_d]:
        (d/"images").mkdir(parents=True, exist_ok=True)
        (d/"labels").mkdir(parents=True, exist_ok=True)

    # ── Build pairs with dataset-tagged names ──
    pairs = []
    per_ds_count = {}
    for ds in train_sets:
        ds_tag = ds.replace(" ", "_")
        cnt = 0
        for img in (BASE/ds/"images").glob("*.*"):
            lbl = BASE/ds/"labels"/(img.stem + ".txt")
            if lbl.exists():
                new_stem = f"{ds_tag}__{img.stem}"     # ← dataset prefix
                pairs.append((img, lbl, new_stem))
                cnt += 1
        per_ds_count[ds] = cnt

    print(f"  [make_val_split] train_sets = {train_sets}")
    for ds, cnt in per_ds_count.items():
        print(f"    {ds:22s} {cnt:5d} valid pairs")
    print(f"    {'TOTAL':22s} {len(pairs):5d} valid pairs")

    # ── Split ──
    n_val = max(1, int(len(pairs) * VAL_FRAC))
    val_pairs   = random.sample(pairs, n_val)
    val_new     = {s for _, _, s in val_pairs}
    train_pairs = [(i, l, s) for i, l, s in pairs if s not in val_new]

    print(f"    → val={len(val_pairs)}, train={len(train_pairs)}")

    # ── Symlink val ──
    for img, lbl, new_stem in val_pairs:
        vimg = val_dir/"images"/(new_stem + img.suffix)
        vlbl = val_dir/"labels"/(new_stem + ".txt")
        if not vimg.exists(): os.symlink(img, vimg)
        if not vlbl.exists(): os.symlink(lbl, vlbl)

    # ── Symlink train_only ──
    for img, lbl, new_stem in train_pairs:
        timg = train_only_d/"images"/(new_stem + img.suffix)
        tlbl = train_only_d/"labels"/(new_stem + ".txt")
        if not timg.exists(): os.symlink(img, timg)
        if not tlbl.exists(): os.symlink(lbl, tlbl)

    print(f"  built: train={len(train_pairs)}, val={len(val_pairs)} (disjoint, prefixed)")



In [ ]:
# Resume: skip folds already in CSV
done = set()
if SUMMARY_CSV.exists():
    done = set(pd.read_csv(SUMMARY_CSV)["test_on"].tolist())
    print(f"Resuming — already done: {done or 'none'}")

for test_ds in DATASETS:
    if test_ds in done:
        print(f"⏭  skipping {test_ds} (already in CSV)")
        continue

    train_sets = [d for d in DATASETS if d != test_ds]
    fold_dir = FOLDS / f"test_{test_ds.replace(' ', '_')}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    # Build val subset + YAMLs
    # Build disjoint train + val
    val_dir = fold_dir/"val"
    make_val_split(train_sets, fold_dir)          #  passes fold_dir

    with open(fold_dir/"train.yaml", "w") as f:
      yaml.dump({
          "train": str(fold_dir/"train_only"/"images"),          #  disjoint symlink set
          "val":   str(fold_dir/"val"/"images"),
          "nc": 1, "names": ["polyp"],
      }, f)

    with open(fold_dir/"test.yaml", "w") as f:
        yaml.dump({
            "train": str(BASE/test_ds/"images"),
            "val":   str(BASE/test_ds/"images"),
            "test":  str(BASE/test_ds/"images"),
            "nc": 1, "names": ["polyp"],
        }, f)

    print(f"\n{'='*70}\nFOLD test={test_ds}  train={train_sets}\n{'='*70}")

  [make_val_split] train_sets = ['CVC-colonDB', 'ETIS-LaribPolypDB', 'hyperkvasir']
    CVC-colonDB              379 valid pairs
    ETIS-LaribPolypDB        156 valid pairs
    hyperkvasir             1000 valid pairs
    TOTAL                   1535 valid pairs
    → val=230, train=1305
  built: train=1305, val=230 (disjoint, prefixed)

FOLD test=CVC-clinicDB  train=['CVC-colonDB', 'ETIS-LaribPolypDB', 'hyperkvasir']
  [make_val_split] train_sets = ['CVC-clinicDB', 'ETIS-LaribPolypDB', 'hyperkvasir']
    CVC-clinicDB             612 valid pairs
    ETIS-LaribPolypDB        156 valid pairs
    hyperkvasir             1000 valid pairs
    TOTAL                   1768 valid pairs
    → val=265, train=1503
  built: train=1503, val=265 (disjoint, prefixed)

FOLD test=CVC-colonDB  train=['CVC-clinicDB', 'ETIS-LaribPolypDB', 'hyperkvasir']
  [make_val_split] train_sets = ['CVC-clinicDB', 'CVC-colonDB', 'hyperkvasir']
    CVC-clinicDB             612 valid pairs
    CVC-colonDB              

In [ ]:
from pathlib import Path
import yaml

BASE  = Path("/content/polyp_datasets_local")
FOLDS = Path("/content/folds")
DATASETS = ["CVC-clinicDB", "CVC-colonDB", "ETIS-LaribPolypDB", "hyperkvasir"]

def count_imgs(path):
    p = Path(path)
    return len(list(p.glob("*.*"))) if p.exists() else 0

print(f"{'Fold':<6} {'Test set':<22} {'Train imgs':>11} {'Val imgs':>9} {'Test imgs':>10}  {'Train datasets'}")
print("-" * 110)


print("-" * 110)
print(f"Total images across all datasets: {sum(count_imgs(BASE/d/'images') for d in DATASETS)}")

Fold   Test set                Train imgs  Val imgs  Test imgs  Train datasets
--------------------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------------------
Total images across all datasets: 2147


In [ ]:
# @title
from pathlib import Path
train_files = list((Path("/content/folds/test_CVC-clinicDB/train_only/images")).iterdir())
prefixed = sum(1 for f in train_files if "__" in f.name)
print(f"Fold 1: {prefixed}/{len(train_files)} prefixed files")

train_files_2 = list((Path("/content/folds/test_CVC-colonDB/train_only/images")).iterdir())
prefixed_2 = sum(1 for f in train_files_2 if "__" in f.name)
print(f"Fold 2: {prefixed_2}/{len(train_files_2)} prefixed files")

Fold 1: 1305/1305 prefixed files
Fold 2: 1503/1503 prefixed files


In [ ]:
import os
files = os.listdir("/content/folds/test_CVC-clinicDB/train_only/images")
prefixes = set(f.split("__")[0] for f in files if "__" in f)
print(f"Prefixes in train_only: {sorted(prefixes)}")

In [ ]:
for i, test_ds in enumerate(DATASETS, 1):

    fold_dir   = FOLDS / f"test_{test_ds.replace(' ', '_')}"
    train_sets = [d for d in DATASETS if d != test_ds]
    fold       = fold_dir

    ty = fold / "train.yaml"
    sy = fold / "test.yaml"

    if not ty.exists() or not sy.exists():
        print(f"{i:<6} {test_ds:<22} (not built)")
        continue

    with open(ty) as f: tcfg = yaml.safe_load(f)
    with open(sy) as f: scfg = yaml.safe_load(f)

    train_paths = tcfg["train"] if isinstance(tcfg["train"], list) else [tcfg["train"]]
    train_total = sum(count_imgs(p) for p in train_paths)
    val_total   = count_imgs(tcfg["val"])
    test_total  = count_imgs(scfg["test"])

    print(f"\n{'='*70}")
    print(f"FOLD {i}: test={test_ds}")
    print(f"  train={train_sets}")
    print(f"  train_imgs={train_total}  val_imgs={val_total}  test_imgs={test_total}")
    print(f"{'='*70}")

    # ── TRAIN ──
    model = YOLO(MODEL)
    model.train(
        data=str(fold_dir/"train.yaml"),
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
        optimizer="AdamW", lr0=0.001, lrf=0.01, weight_decay=0.0005,
        momentum=0.937, warmup_epochs=3.0,
        box=7.5, cls=0.5, dfl=1.5,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=10.0, translate=0.1, scale=0.2,
        fliplr=0.5, mosaic=1.0, mixup=0.05, close_mosaic=10,
        workers=2, cache=False, amp=True,
        val=True, plots=True,
        project=str(RESULTS/"runs"),
        name=f"train_{test_ds.replace(' ', '_')}",
        exist_ok=True, verbose=False,
    )

    # ── TEST on held-out ──
    best = RESULTS/"runs"/f"train_{test_ds.replace(' ', '_')}"/"weights"/"best.pt"
    m = YOLO(str(best)).val(
        data=str(fold_dir/"test.yaml"),
        split="test", imgsz=IMGSZ,
        project=str(RESULTS/"runs"),
        name=f"eval_{test_ds.replace(' ', '_')}",
        exist_ok=True, verbose=False,
    )

    row = {
        "train_on":  " + ".join(sorted(train_sets)),
        "test_on":   test_ds,
        "mAP50-95":  round(m.box.map,   4),
        "mAP50":     round(m.box.map50, 4),
        "precision": round(m.box.mp,    4),
        "recall":    round(m.box.mr,    4),
    }
    print("→", row)

    pd.DataFrame([row]).to_csv(
        SUMMARY_CSV, mode="a", header=not SUMMARY_CSV.exists(), index=False
    )
    print(f"saved → {SUMMARY_CSV}")


FOLD 1: test=CVC-clinicDB
  train=['CVC-colonDB', 'ETIS-LaribPolypDB', 'hyperkvasir']
  train_imgs=1305  val_imgs=230  test_imgs=612
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/folds/test_CVC-clinicDB/train.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, m

In [ ]:
import pandas as pd
df = pd.read_csv(SUMMARY_CSV)
print("="*70)
print(f"LODO SUMMARY — {MODEL}")
print("="*70)
print(df.to_string(index=False))
print("\n" + "-"*70)
print(f"MEAN     mAP50-95={df['mAP50-95'].mean():.4f}  "
      f"mAP50={df['mAP50'].mean():.4f}  "
      f"P={df['precision'].mean():.4f}  R={df['recall'].mean():.4f}")
print(f"STD      mAP50-95={df['mAP50-95'].std():.4f}   "
      f"mAP50={df['mAP50'].std():.4f}   "
      f"P={df['precision'].std():.4f}  R={df['recall'].std():.4f}")
print(f"\nCSV saved at: {SUMMARY_CSV}")

LODO SUMMARY — yolo11s.pt
                                      train_on           test_on  mAP50-95  mAP50  precision  recall
 CVC-colonDB + ETIS-LaribPolypDB + hyperkvasir      CVC-clinicDB    0.6336 0.8934     0.8869  0.8374
CVC-clinicDB + ETIS-LaribPolypDB + hyperkvasir       CVC-colonDB    0.5294 0.8086     0.7754  0.7414
      CVC-clinicDB + CVC-colonDB + hyperkvasir ETIS-LaribPolypDB    0.5240 0.7484     0.8514  0.6479
CVC-clinicDB + CVC-colonDB + ETIS-LaribPolypDB       hyperkvasir    0.5622 0.7885     0.7739  0.7750

----------------------------------------------------------------------
MEAN     mAP50-95=0.5623  mAP50=0.8097  P=0.8219  R=0.7504
STD      mAP50-95=0.0504   mAP50=0.0611   P=0.0565  R=0.0791

CSV saved at: /content/drive/MyDrive/Computer Vision Term Project/lodo_results/lodo_summary.csv


#2.Specular Highlights Error Analysis

In [ ]:
!pip install -q ultralytics pyyaml pandas

In [ ]:
!pip install -q opencv-python-headless
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive
drive.mount('/content/drive')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

BASE    = Path("/content/drive/MyDrive/Computer Vision Term Project")
RESULTS = Path("/content/drive/MyDrive/Computer Vision Term Project/lodo_results")
DATASETS = ["CVC-clinicDB", "CVC-colonDB", "ETIS-LaribPolypDB", "hyperkvasir"]

IOU_THRESH   = 0.5
CONF_THRESH  = 0.25
V_THRESH     = 220    # bright pixels
S_THRESH     = 50     # low saturation → highlight (near white)

In [ ]:
# ────────── HELPERS ──────────
def load_gt_xyxy(lbl_path, W, H):
    boxes = []
    if not lbl_path.exists():
        return np.zeros((0, 4))
    for line in lbl_path.read_text().strip().split("\n"):
        p = line.split()
        if len(p) < 5:
            continue
        _, xc, yc, w, h = map(float, p[:5])
        boxes.append([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H])
    return np.array(boxes) if boxes else np.zeros((0, 4))

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a = a[:, None, :]; b = b[None, :, :]
    x1 = np.maximum(a[...,0], b[...,0]); y1 = np.maximum(a[...,1], b[...,1])
    x2 = np.minimum(a[...,2], b[...,2]); y2 = np.minimum(a[...,3], b[...,3])
    inter = np.clip(x2-x1,0,None) * np.clip(y2-y1,0,None)
    aa = (a[...,2]-a[...,0]) * (a[...,3]-a[...,1])
    ab = (b[...,2]-b[...,0]) * (b[...,3]-b[...,1])
    return inter / np.maximum(aa + ab - inter, 1e-9)

def highlight_frac(img_bgr, xyxy):
    """Fraction of GT-box pixels that are specular highlights."""
    x1,y1,x2,y2 = map(int, xyxy)
    H, W = img_bgr.shape[:2]
    x1, y1 = max(0,x1), max(0,y1)
    x2, y2 = min(W,x2), min(H,y2)
    if x2 <= x1 or y2 <= y1:
        return np.nan
    crop = img_bgr[y1:y2, x1:x2]
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    V = hsv[:,:,2].astype(np.float32)
    S = hsv[:,:,1].astype(np.float32)
    mask = (V > V_THRESH) & (S < S_THRESH)
    return float(mask.sum()) / mask.size if mask.size > 0 else np.nan

In [ ]:
import shutil
from pathlib import Path
SRC   = Path("/content/drive/MyDrive/Computer Vision Term Project/polyp_datasets")
LOCAL = Path("/content/polyp_datasets_local")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
shutil.copytree(SRC, LOCAL)
print("Recopied local data")

Recopied local data


In [ ]:
from pathlib import Path
BASE = Path("/content/polyp_datasets_local")
DATASETS = ["CVC-clinicDB", "CVC-colonDB", "ETIS-LaribPolypDB", "hyperkvasir"]

for ds in DATASETS:
    img_dir = BASE/ds/"images"
    lbl_dir = BASE/ds/"labels"
    n_img = len(list(img_dir.glob("*.*"))) if img_dir.exists() else -1
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else -1
    print(f"{ds:22s}  images={n_img:5d}  labels={n_lbl:5d}")

CVC-clinicDB            images=  612  labels=  612
CVC-colonDB             images=  379  labels=  379
ETIS-LaribPolypDB       images=  156  labels=  156
hyperkvasir             images= 1000  labels= 1000


In [ ]:
# ────────── PER-FOLD ANALYSIS ──────────
records = []
for test_ds in DATASETS:
    name = test_ds.replace(' ', '_')
    weights = RESULTS / "runs" / f"train_{name}" / "weights" / "best.pt"
    if not weights.exists():
        print(f"⏭  no weights for {test_ds} — skipping")
        continue
    model = YOLO(str(weights))

    img_dir = BASE / test_ds / "images"
    lbl_dir = BASE / test_ds / "labels"

    for img_path in sorted(img_dir.glob("*.*")):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue
        im = cv2.imread(str(img_path))
        if im is None:
            continue
        H, W = im.shape[:2]
        gt = load_gt_xyxy(lbl_path, W, H)
        if len(gt) == 0:
            continue

        r = model.predict(str(img_path), conf=CONF_THRESH, verbose=False)[0]
        preds = r.boxes.xyxy.cpu().numpy() if r.boxes is not None and len(r.boxes) else np.zeros((0,4))

        iou = iou_matrix(gt, preds)
        detected = (iou.max(axis=1) >= IOU_THRESH) if len(preds) else np.zeros(len(gt), bool)

        for i, box in enumerate(gt):
            records.append({
                "dataset": test_ds,
                "image": img_path.name,
                "gt_idx": i,
                "detected": bool(detected[i]),
                "highlight_frac": highlight_frac(im, box),
                "gt_area_frac": float((box[2]-box[0])*(box[3]-box[1])) / (W*H),
            })

df = pd.DataFrame(records).dropna(subset=["highlight_frac"])
df["highlight_frac"] = df["highlight_frac"].clip(0, 1)
print(f"Total GT polyps analyzed: {len(df)}")



Total GT polyps analyzed: 2264


In [ ]:
# ────────── SPLIT + COMPARE ──────────
# Fixed threshold (0.02 = 2% of polyp area is specular highlight)
HL_THR = 0.02

rows = []
for ds in df["dataset"].unique():
    sub = df[df.dataset == ds]
    low  = sub[sub.highlight_frac <= HL_THR]
    high = sub[sub.highlight_frac >  HL_THR]
    for label, g in [("low", low), ("high", high)]:
        if len(g) == 0:
            continue
        recall = g["detected"].mean()
        rows.append({
            "dataset": ds, "highlight": label,
            "n": len(g),
            "recall": round(recall, 3),
            "miss_rate": round(1-recall, 3),
        })

hl_df = pd.DataFrame(rows)
print("\n=== Specular highlight analysis ===")
print(hl_df.pivot(index="dataset", columns="highlight",
                  values=["n","recall","miss_rate"]).to_string())



=== Specular highlight analysis ===
                       n        recall        miss_rate       
highlight           high    low   high    low      high    low
dataset                                                       
CVC-clinicDB        74.0  572.0  0.838  0.865     0.162  0.135
CVC-colonDB         41.0  338.0  0.829  0.719     0.171  0.281
ETIS-LaribPolypDB   50.0  118.0  0.780  0.636     0.220  0.364
hyperkvasir        287.0  784.0  0.805  0.792     0.195  0.208


In [ ]:
# ── SIZE CONFOUND CHECK ─────────────────────────────
print("\n=== Polyp size by dataset ===")
print(df.groupby("dataset")["gt_area_frac"].describe()[["count","mean","50%","min","max"]].round(4))

print("\n=== Mean polyp size by glare group (per dataset) ===")
for ds in df["dataset"].unique():
    sub = df[df.dataset == ds]
    if len(sub) == 0:
        continue
    g = sub.groupby(sub.highlight_frac > HL_THR)["gt_area_frac"].agg(["count","mean"]).round(4)
    g.index = ["low_glare", "high_glare"] if len(g) == 2 else g.index
    print(f"\n--- {ds} ---")
    print(g)
hl_df.to_csv(RESULTS/"analysis_highlights.csv", index=False)
print(f"\n saved → {RESULTS/'analysis_highlights.csv'}")


=== Polyp size by dataset ===
                    count    mean     50%     min     max
dataset                                                  
CVC-clinicDB        646.0  0.1314  0.0889  0.0023  0.7174
CVC-colonDB         379.0  0.1085  0.0491  0.0036  0.8142
ETIS-LaribPolypDB   168.0  0.0648  0.0277  0.0013  0.4074
hyperkvasir        1071.0  0.2125  0.1503  0.0000  0.9983

=== Mean polyp size by glare group (per dataset) ===

--- CVC-clinicDB ---
            count    mean
low_glare     572  0.1274
high_glare     74  0.1621

--- CVC-colonDB ---
            count    mean
low_glare     338  0.1072
high_glare     41  0.1191

--- ETIS-LaribPolypDB ---
            count    mean
low_glare     118  0.0709
high_glare     50  0.0503

--- hyperkvasir ---
            count    mean
low_glare     784  0.1989
high_glare    287  0.2495

💾 saved → /content/drive/MyDrive/Computer Vision Term Project/lodo_results/analysis_highlights.csv


#3.Flat / Sessile Polyp Error Analysis

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FLAT / SESSILE POLYP ERROR ANALYSIS
# ══════════════════════════════════════════════════════════════════
# GOAL: Test whether YOLO misses flat/sessile polyps more than other polyps,
#       across all LODO folds.
#
#
# PIPELINE:
#   1. Train small CNN on Kvasir-Sessile → binary: sessile/flat (1) vs other (0).
#
#   2. For each LODO fold, for each GT polyp box:
#        - Crop the GT box from the test image
#        - Run the trained CNN → predicted morphology label + confidence
#   3. Each GT polyp now has: detected (YOLO), morph_label (CNN), morph_prob.
#   4. Group polyps by predicted morphology per dataset.
#   5. Compare recall: sessile/flat vs other.
#      If recall(sessile/flat) < recall(other) → flat polyps are missed more.
#      Recall gap = recall(other) − recall(sessile/flat). Positive = flat harder.


#

In [ ]:
!pip install -q ultralytics pyyaml pandas

In [ ]:
# ══════════════════════════════════════════════════════════════════
# FLAT / SESSILE POLYP ERROR ANALYSIS — PolypDB_INS datset
# ══════════════════════════════════════════════════════════════════
# unzip from Drive → find JSON + images → train classifier →
# apply to LODO folds → compare sessile/flat vs other recall.


import json, os, random, zipfile
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#  ───────────────────────────────────────────────
ZIP_PATH = Path("/content/drive/MyDrive/Computer Vision Term Project/use dataset/PolypDB_INSdatasets_V1.1 (1).zip")
BASE     = Path("/content/polyp_datasets_local")
RESULTS  = Path("/content/drive/MyDrive/Computer Vision Term Project/lodo_results")
# ─────────────────────────────────────────────────────────────────

EXTRACT      = Path("/content/PolypDB_INS")
CKPT_PATH    = RESULTS / "morphology_classifier.pt"
DATASETS     = ["CVC-clinicDB", "CVC-colonDB", "ETIS-LaribPolypDB", "hyperkvasir"]
IOU_THRESH, CONF_THRESH = 0.5, 0.25
CLS_SIZE, EPOCHS_CLS    = 224, 30
RESULTS.mkdir(parents=True, exist_ok=True)



In [ ]:
# ══════════════════════════════════════════════════════════════════
# 0. UNZIP ( skips if already extracted)
# ══════════════════════════════════════════════════════════════════
if not EXTRACT.exists() or not any(EXTRACT.iterdir()):
    print(f"Unzipping {ZIP_PATH.name} → {EXTRACT} …")
    EXTRACT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT)
    print("unzipped")
else:
    print(f"already extracted at {EXTRACT}")

already extracted at /content/PolypDB_INS


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 1. FIND JSON + IMAGES ROOT
# ══════════════════════════════════════════════════════════════════
json_candidates = sorted(EXTRACT.rglob("*.json"),
                         key=lambda p: (p.name != "polypdb_ins.json", len(p.parts)))
if not json_candidates:
    raise FileNotFoundError(f"No JSON found under {EXTRACT}")
JSON_PATH = json_candidates[0]
print(f"Using JSON: {JSON_PATH}")

# Images root = folder whose parent contains a child folder named 'images',
# OR the folder that contains subdatasets directly.
img_root_candidates = []
for d in EXTRACT.rglob("*"):
    if not d.is_dir(): continue
    if d.name.lower() == "images":
        img_root_candidates.append(d.parent)
for d in EXTRACT.rglob("*"):
    if not d.is_dir(): continue
    kids = {c.name for c in d.iterdir() if c.is_dir()}
    if kids & {"Kvasir-SEG", "CVC-ClinicDB", "CVC-ColonDB",
               "ETIS-LaribPolypDB", "PolypGen", "NeoPolyp-Small"}:
        img_root_candidates.append(d)
if not img_root_candidates:
    raise FileNotFoundError(f"No images/ folder found under {EXTRACT}")
DATASET_ROOT = sorted(img_root_candidates, key=lambda p: len(p.parts))[0]
print(f"Using DATASET_ROOT: {DATASET_ROOT}")
print("  children:", [c.name for c in DATASET_ROOT.iterdir() if c.is_dir()][:10])

Using JSON: /content/PolypDB_INS/datasets/polypdb_ins.json
Using DATASET_ROOT: /content/PolypDB_INS/datasets
  children: ['images', 'images-mask']


In [ ]:

# ══════════════════════════════════════════════════════════════════
# 2. PARSE JSON
# ══════════════════════════════════════════════════════════════════
with open(JSON_PATH) as f:
    db = json.load(f)

cat_id_to_name = {c["id"]: c["name"].lower() for c in db["categories"]}
print(f"\nCategories: {cat_id_to_name}")

SESSILE_NAMES = {"sessile", "flat", "is", "iia", "ii"}
def to_binary(name): return 1 if any(k in name.lower() for k in SESSILE_NAMES) else 0

def resolve_image(file_name):
    rel = file_name.lstrip("./")
    candidates = [
        DATASET_ROOT / rel,
        DATASET_ROOT / "images" / Path(rel).name,
        DATASET_ROOT / Path(rel).name,
        EXTRACT / rel,
        EXTRACT / "images" / Path(rel).name,
    ]
    for c in candidates:
        if c.exists(): return c
    return None

image_id_to_file = {}
missing = 0
for im in db["images"]:
    p = resolve_image(im["file_name"])
    image_id_to_file[im["id"]] = p
    if p is None: missing += 1
print(f"Images resolved: {len(image_id_to_file)-missing}/{len(image_id_to_file)}  "
      f"(missing={missing})")

# show one example if missing
if missing > 0:
    ex = next(im for im in db["images"]
              if image_id_to_file[im["id"]] is None)
    print(f"  Example unresolved file_name: {ex['file_name']}")
    print(f"  Tried: {DATASET_ROOT / ex['file_name'].lstrip('./')}")

# Build polyp items
items = []
for ann in db["annotations"]:
    p = image_id_to_file.get(ann["image_id"])
    if p is None: continue
    cat = cat_id_to_name.get(ann["category_id"], "unknown")
    x, y, w, h = ann["bbox"]
    items.append((p, [x, y, x+w, y+h], to_binary(cat)))

n_pos = sum(1 for _,_,l in items if l == 1)
n_neg = sum(1 for _,_,l in items if l == 0)
print(f"\nPolypDB_INS: {len(items)} polyps — "
      f"{n_pos} sessile/flat (pos), {n_neg} pedunculated/other (neg)")
if len(items) == 0:
    raise RuntimeError("No items — see unresolved path printed above.")


Categories: {1: 'pedunculated', 2: 'sessile'}
Images resolved: 4403/4403  (missing=0)

PolypDB_INS: 4918 polyps — 4566 sessile/flat (pos), 352 pedunculated/other (neg)


In [ ]:

# ══════════════════════════════════════════════════════════════════
# 3.  Dataset
# ══════════════════════════════════════════════════════════════════

class PolypCropDataset(Dataset):
    def __init__(self, items, size=CLS_SIZE):
        self.items, self.size = items, size
        self.tfm_train = transforms.Compose([
            transforms.ToPILImage(), transforms.Resize((size, size)),
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        self.tfm_val = transforms.Compose([
            transforms.ToPILImage(), transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        self.training = True
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        img_path, bbox, y = self.items[i]
        im = cv2.imread(str(img_path))
        if im is None:
            crop = np.zeros((self.size, self.size, 3), dtype=np.uint8)
        else:
            H, W = im.shape[:2]
            x1, y1, x2, y2 = bbox
            # ── PAD 40% so the stalk (defining feature) is included ──
            pad_x = 0.40 * (x2 - x1)
            pad_y = 0.40 * (y2 - y1)
            x1 = max(0, int(round(x1 - pad_x))); y1 = max(0, int(round(y1 - pad_y)))
            x2 = min(W, int(round(x2 + pad_x))); y2 = min(H, int(round(y2 + pad_y)))
            crop = (np.zeros((self.size, self.size, 3), dtype=np.uint8)
                    if x2 <= x1 or y2 <= y1
                    else cv2.cvtColor(im[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
        t = self.tfm_train if self.training else self.tfm_val
        return t(crop), torch.tensor(y, dtype=torch.long)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4. Train
# ══════════════════════════════════════════════════════════════════
def train_classifier():
    ds = PolypCropDataset(items)
    n_val = max(1, int(0.2 * len(ds))); n_tr = len(ds) - n_val
    tr_ds, va_ds = torch.utils.data.random_split(
        ds, [n_tr, n_val], generator=torch.Generator().manual_seed(SEED))
    tr_loader = DataLoader(tr_ds, batch_size=32, shuffle=True,  num_workers=2)
    va_loader = DataLoader(va_ds, batch_size=32, shuffle=False, num_workers=2)


    from torchvision.models import resnet18, ResNet18_Weights
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(512, 2)
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=3e-4)

    counts  = np.bincount([y for _,_,y in items], minlength=2)
    weights = torch.tensor(counts.sum() / (2 * np.maximum(counts, 1)),
                           dtype=torch.float32).to(DEVICE)
    print(f"Class counts: {counts.tolist()}  weights: {weights.tolist()}")
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    best_acc, best_state = 0.0, None
    for epoch in range(EPOCHS_CLS):
        ds.training = True; model.train()
        tot = corr = 0; lsum = 0.0
        for x, y in tr_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            logits = model(x); loss = loss_fn(logits, y)
            loss.backward(); opt.step()
            lsum += loss.item()*len(y); corr += (logits.argmax(1)==y).sum().item(); tot += len(y)
        ds.training = False; model.eval()
        vtot = vcorr = 0; pc = {0:[0,0], 1:[0,0]}
        with torch.no_grad():
            for x, y in va_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                pred = model(x).argmax(1)
                vcorr += (pred==y).sum().item(); vtot += len(y)
                for c in [0,1]:
                    m = (y==c); pc[c][0] += (pred[m]==y[m]).sum().item(); pc[c][1] += m.sum().item()
        va = vcorr/max(vtot,1); a0 = pc[0][0]/max(pc[0][1],1); a1 = pc[1][0]/max(pc[1][1],1)
        print(f"epoch {epoch+1:2d} | loss {lsum/tot:.3f} tr {corr/tot:.3f} | "
              f"val {va:.3f}  [neg {a0:.3f} n={pc[0][1]} | pos {a1:.3f} n={pc[1][1]}]")
        if va > best_acc:
            best_acc = va; best_state = {k:v.detach().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state)
    torch.save(best_state, CKPT_PATH)
    print(f"\n saved → {CKPT_PATH}  (best val acc = {best_acc:.3f})")
    return model


# ══════════════════════════════════════════════════════════════════
# 5. LODO helpers + inference
# ══════════════════════════════════════════════════════════════════
def load_gt_xyxy(lbl_path, W, H):
    boxes = []
    if not lbl_path.exists(): return np.zeros((0, 4))
    for line in lbl_path.read_text().strip().split("\n"):
        p = line.split()
        if len(p) < 5: continue
        _, xc, yc, w, h = map(float, p[:5])
        boxes.append([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H])
    return np.array(boxes) if boxes else np.zeros((0, 4))

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    a = a[:, None, :]; b = b[None, :, :]
    x1 = np.maximum(a[...,0], b[...,0]); y1 = np.maximum(a[...,1], b[...,1])
    x2 = np.minimum(a[...,2], b[...,2]); y2 = np.minimum(a[...,3], b[...,3])
    inter = np.clip(x2-x1,0,None)*np.clip(y2-y1,0,None)
    aa = (a[...,2]-a[...,0])*(a[...,3]-a[...,1]); ab = (b[...,2]-b[...,0])*(b[...,3]-b[...,1])
    return inter / np.maximum(aa + ab - inter, 1e-9)

_tfm_infer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
def classify_crop(model, img_bgr, xyxy, size=CLS_SIZE):

    x1,y1,x2,y2 = xyxy
    H, W = img_bgr.shape[:2]
    pad_x = 0.40 * (x2 - x1); pad_y = 0.40 * (y2 - y1)
    x1 = max(0, int(round(x1 - pad_x))); y1 = max(0, int(round(y1 - pad_y)))
    x2 = min(W, int(round(x2 + pad_x))); y2 = min(H, int(round(y2 + pad_y)))
    if x2 <= x1 or y2 <= y1: return 0, np.nan
    crop = cv2.cvtColor(img_bgr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)

    crop = cv2.resize(crop, (size, size))
    t = _tfm_infer(crop).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(t), dim=1)[0].cpu().numpy()
    return int(probs.argmax()), float(probs[1])

def run_lodo_morphology():

    if CKPT_PATH.exists():
      print(f" loading existing classifier: {CKPT_PATH}")
      from torchvision.models import resnet18
      morph = resnet18(weights=None)
      morph.fc = nn.Linear(512, 2)
      morph = morph.to(DEVICE)
      morph.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    else:
        morph = train_classifier()
    morph.eval()

    records = []
    for test_ds in DATASETS:
        name   = test_ds.replace(' ', '_')
        w_path = RESULTS / "runs" / f"train_{name}" / "weights" / "best.pt"
        if not w_path.exists():
            print(f"⏭  no YOLO weights for {test_ds}"); continue
        model   = YOLO(str(w_path))
        img_dir = BASE / test_ds / "images"
        lbl_dir = BASE / test_ds / "labels"
        n_img = n_polyp = 0
        for img_path in sorted(img_dir.glob("*.*")):
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if not lbl_path.exists(): continue
            im = cv2.imread(str(img_path))
            if im is None: continue
            H, W = im.shape[:2]
            gt = load_gt_xyxy(lbl_path, W, H)
            if len(gt) == 0: continue
            r = model.predict(str(img_path), conf=CONF_THRESH, verbose=False)[0]
            preds = r.boxes.xyxy.cpu().numpy() if r.boxes is not None and len(r.boxes) else np.zeros((0,4))
            iou = iou_matrix(gt, preds)
            detected = (iou.max(axis=1) >= IOU_THRESH) if len(preds) else np.zeros(len(gt), bool)
            for i, box in enumerate(gt):
                ml, mp = classify_crop(morph, im, box)
                records.append({"dataset": test_ds, "image": img_path.name, "gt_idx": i,
                                "detected": bool(detected[i]),
                                "morph_label": ml, "morph_prob": mp,
                                "gt_area_frac": float((box[2]-box[0])*(box[3]-box[1]))/(W*H)})
                n_polyp += 1
            n_img += 1
        print(f"{test_ds}: {n_img} imgs, {n_polyp} polyps classified")
    return pd.DataFrame(records)

# ══════════════════════════════════════════════════════════════════
# 6. Analyze
# ══════════════════════════════════════════════════════════════════
def analyze_morphology(df):
    if len(df) == 0:
        raise RuntimeError("No polyps classified.")
    rows = []
    for ds in df["dataset"].unique():
        sub = df[df.dataset==ds]
        for lbl, mask in [("sessile/flat", sub.morph_label==1), ("other", sub.morph_label==0)]:
            g = sub[mask]
            if len(g)==0: continue
            r = g["detected"].mean()
            rows.append({"dataset": ds, "morphology": lbl, "n": len(g),
                         "recall": round(r,3), "miss_rate": round(1-r,3),
                         "mean_confidence": round(g["morph_prob"].mean(),3)})
    mdf = pd.DataFrame(rows)
    print("\n=== Morphology-stratified detection ===")
    print(mdf.pivot(index="dataset", columns="morphology",
                    values=["n","recall","miss_rate"]).to_string())
    print("\n=== Recall gap: other − sessile/flat  (positive = flat harder) ===")
    for ds in df["dataset"].unique():
        s = mdf[(mdf.dataset==ds)&(mdf.morphology=="sessile/flat")]
        o = mdf[(mdf.dataset==ds)&(mdf.morphology=="other")]
        if len(s)==0 or len(o)==0:
            print(f"  {ds:22s}  one group empty — skip"); continue
        print(f"  {ds:22s}  gap = {o.recall.iloc[0]-s.recall.iloc[0]:+.3f}")
    mdf.to_csv(RESULTS/"analysis_morphology.csv", index=False)
    print(f"\n saved → {RESULTS/'analysis_morphology.csv'}")
    return mdf

In [ ]:

# ══════════════════════════════════════════════════════════════════
# 7. RUN
# ══════════════════════════════════════════════════════════════════
df_morph = run_lodo_morphology()
mdf      = analyze_morphology(df_morph)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Class counts: [352, 4566]  weights: [6.985795497894287, 0.5385457873344421]
epoch  1 | loss 0.564 tr 0.718 | val 0.791  [neg 0.640 n=75 | pos 0.804 n=908]
epoch  2 | loss 0.433 tr 0.779 | val 0.865  [neg 0.653 n=75 | pos 0.882 n=908]
epoch  3 | loss 0.389 tr 0.810 | val 0.745  [neg 0.880 n=75 | pos 0.733 n=908]
epoch  4 | loss 0.387 tr 0.828 | val 0.835  [neg 0.787 n=75 | pos 0.839 n=908]
epoch  5 | loss 0.352 tr 0.825 | val 0.868  [neg 0.813 n=75 | pos 0.872 n=908]
epoch  6 | loss 0.309 tr 0.856 | val 0.848  [neg 0.813 n=75 | pos 0.851 n=908]
epoch  7 | loss 0.287 tr 0.874 | val 0.780  [neg 0.933 n=75 | pos 0.768 n=908]
epoch  8 | loss 0.256 tr 0.893 | val 0.879  [neg 0.787 n=75 | pos 0.887 n=908]
epoch  9 | loss 0.245 tr 0.898 | val 0.753  [neg 0.880 n=75 | pos 0.742 n=908]
epoch 10 | loss 0.274 tr 0.879 | val 0.897  [neg 0.760 n=75 | pos 0.909 n=908]
epoch 11 | loss 0.196 tr 0.918 | val 0.883  [neg 0.813 n=75 | pos 0.889 n=908]
epoch 12 | loss 0.197 tr 0.923 | val 0.857  [neg 0.867 